# SilicoJev: resumable RTL/DV decision-model fine-tuning

This notebook is the visible control surface. The actual conversion, training, evaluation, and checkpoint logic lives in Python scripts so a disconnected notebook can be restarted safely.

Model lineage: `convaiinnovations/laya-typed-decisions` → `SilicoJev`. The typed output contract remains `choice`, `noul`, and `score`. The Laya encoder/head trainer remains the primary path; Unsloth is reserved for a separate decoder-model baseline because it does not train this ModernBERT decision-head architecture.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path('/home/jovyan/shared/jevlikeresearch')
SILICOJEV = ROOT / 'silicojev'
DATASET = SILICOJEV / 'dataset'
RAW = DATASET / 'raw'
NORMALIZED = DATASET / 'normalized'
MERGED_5Q = NORMALIZED / 'merged_5q'
TRAINING_DATA = MERGED_5Q
MODEL = ROOT / 'models' / 'laya-typed-decisions'
CHECKPOINTS = SILICOJEV / 'checkpoints'
TRAINING = SILICOJEV / 'training'
EVALUATION = SILICOJEV / 'evaluation'

for path in [DATASET, MODEL, TRAINING]:
    assert path.exists(), f'Missing required path: {path}'
print('SilicoJev root:', SILICOJEV)
print('Starting checkpoint:', MODEL)
print('Normalized data:', NORMALIZED)
print('Merged five-question data:', TRAINING_DATA)
print('Checkpoint directory:', CHECKPOINTS)

In [ ]:
import torch

assert torch.cuda.is_available(), 'CUDA is not available; activate the A100 before training.'
gpu = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(gpu.total_memory / 1024**3, 2))
print('BF16 supported:', torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported(), 'This run is configured for BF16.'

In [ ]:
# The three source-derived questions and two distilled score questions are
# merged by stable record ID. This does not download or merge the separate
# LocalLLaMA benchmark; it only completes SilicoJev's existing records.
RUN_BASE_CONVERSION = False
RUN_QUESTION_MERGE = True
INCLUDE_UNVALIDATED_ORIGEN = False
if RUN_BASE_CONVERSION:
    conversion_cmd = [
        sys.executable, str(TRAINING / 'prepare_dataset.py'),
        '--dataset-root', str(DATASET),
    ]
    if INCLUDE_UNVALIDATED_ORIGEN:
        conversion_cmd.append('--include-origen')
    subprocess.run(conversion_cmd, check=True)

if RUN_QUESTION_MERGE:
    subprocess.run([
        sys.executable, str(TRAINING / 'merge_silicojev_questions.py'),
        '--normalized-dir', str(NORMALIZED),
        '--output-dir', str(MERGED_5Q),
    ], check=True)

summary = json.loads((TRAINING_DATA / 'summary.json').read_text())
print(json.dumps(summary, indent=2))

In [ ]:
# Inspect counts before any GPU training. Bronze records remain identifiable.
for name in ['train.jsonl', 'validation.jsonl', 'test.jsonl']:
    path = TRAINING_DATA / name
    print(name, sum(1 for line in path.open() if line.strip()), path.stat().st_size, 'bytes')

first = json.loads(next(line for line in (TRAINING_DATA / 'train.jsonl').open() if line.strip()))
print('record:', first['id'])
print('source:', first['source'])
print('questions:', list(json.loads(first['questions'])))
print('provenance:', first['provenance'])

In [ ]:
# External corpora are downloaded but intentionally not merged automatically.
# Each source needs a schema adapter, provenance-preserving labels, and replay/evaluation checks.
external = {
    'RTL-BenchLS': RAW / 'github' / 'RTL-BenchLS',
    'Fixbench-RTL': RAW / 'hf' / 'Fixbench-RTL',
    'CVDP-v1.1.0': RAW / 'hf' / 'CVDP',
}
for name, path in external.items():
    files = sum(1 for p in path.rglob('*') if p.is_file()) if path.exists() else 0
    print(f'{name}: {path} | files={files} | present={path.exists()}')
print('These corpora remain raw/audit-only until conversion and project-disjoint split checks pass.')

In [ ]:
# Safe default: do not start GPU training merely by opening/running setup cells.
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    subprocess.run([
        sys.executable, str(TRAINING / 'train_silicojev.py'),
        '--model-dir', str(MODEL),
        '--train', str(TRAINING_DATA / 'train.jsonl'),
        '--validation', str(TRAINING_DATA / 'validation.jsonl'),
        '--output-dir', str(CHECKPOINTS),
        '--epochs', '1',
        '--max-steps', '25',
        '--micro-batch-size', '16',
        '--grad-accum-steps', '4',
        '--checkpoint-every', '10',
        '--resume', 'none',
        '--dtype', 'bf16',
    ], check=True)
else:
    print('Smoke training is disabled. Set RUN_SMOKE_TEST=True after reviewing the converted data.')

In [ ]:
subprocess.run([
    sys.executable, str(TRAINING / 'monitor_checkpoints.py'),
    '--checkpoint-root', str(CHECKPOINTS),
], check=True)

In [ ]:
# Evaluate the latest completed checkpoint. This cell is safe before training.
latest = CHECKPOINTS / 'latest'
if latest.exists() and (latest / 'model.safetensors').exists():
    subprocess.run([
        sys.executable, str(TRAINING / 'evaluate_silicojev.py'),
        '--model-dir', str(latest),
        '--data', str(TRAINING_DATA / 'test.jsonl'),
        '--output', str(EVALUATION / 'latest.json'),
        '--device', 'cuda',
    ], check=True)
else:
    print('No completed SilicoJev checkpoint yet.')

## Evaluation notes

The evaluator now reports exact accuracy, soft accuracy, Brier score, KL divergence, total variation, ECE, latency p50/p95, per-question-type metrics, and per-source metrics. It also writes `evaluation/latest_by_source.json`.

The final training checkpoint fits validation-set temperatures for `choice`, `score`, and `noul` with LBFGS and stores the calibration metadata in `rl_agent_config.json`.

The merged dataset contains `risk` and `urgency` score questions. Their four-level labels are preserved as `codex_pseudo_unverified`; score metrics can be reported for exploratory diagnostics, but must not be treated as final quality or evaluation truth until independently validated. The original three source-derived questions remain unchanged.

The downloaded RTL-BenchLS, Fixbench-RTL, and CVDP corpora are raw/audit-only in this notebook. Do not convert patches, reference answers, or benchmark metadata into pseudo risk/urgency labels.

In [ ]:
# Long run: resume automatically from checkpoints/latest. Keep disabled until smoke metrics are reviewed.
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    subprocess.run([
        sys.executable, str(TRAINING / 'train_silicojev.py'),
        '--model-dir', str(MODEL),
        '--train', str(TRAINING_DATA / 'train.jsonl'),
        '--validation', str(TRAINING_DATA / 'validation.jsonl'),
        '--output-dir', str(CHECKPOINTS),
        '--epochs', '4',
        '--micro-batch-size', '16',
        '--grad-accum-steps', '4',
        '--checkpoint-every', '100',
        '--resume', 'auto',
        '--dtype', 'bf16',
    ], check=True)
else:
    print('Full training is disabled. Review the smoke-test loss, calibration, and source balance first.')